---
# Resumen de la clase anterior

## Texto a bytes

- Un `str` es una secuencia de caracteres; un archivo guarda bytes.
- El encoding es la regla para pasar de uno a otro: `.encode()` (texto → bytes) y `.decode()` (bytes → texto).
- `utf-8`: estándar actual, cualquier carácter, 1 a 4 bytes por carácter.
- `cp1252` / `latin-1`: un byte por carácter, solo ~256 caracteres (típico de Windows).

In [40]:
texto = "Ñorquinco"
b = texto.encode("utf-8")  # bytes
print(list(b))
# b = texto.encode("latin-1")
# print(list(b))

[195, 145, 111, 114, 113, 117, 105, 110, 99, 111]


In [44]:
print(b.decode("utf-8"))    # bien
print(b.decode("cp1252"))   # mal

Ñorquinco
Ã‘orquinco


In [47]:
caracter = 'o'
ord(caracter)

111

In [48]:
list(caracter.encode('utf-8' ))

[111]

## Unicode

Cada carácter tiene un número (`ord`). Se escribe en hexadecimal como `U+XXXX`.

In [49]:
for c in "abc":
    print(c, ord(c), f"\\u{ord(c):04X}")

a 97 \u0061
b 98 \u0062
c 99 \u0063


In [50]:
print("\u0061\u0062\u0063")

abc


In [52]:
print(r"\u0061\u0062\u0063")

\u0061\u0062\u0063


In [54]:
no_imprimibles = '\n\t\\'
print(no_imprimibles)
print(repr(no_imprimibles))


	\
'\n\t\\'


## Secuencias de escape

`\n`, `\t`, `\\`, `\"` valen **un** carácter cada una.
Cuidado con rutas de Windows: `"C:\nuevo"` tiene un salto de línea. Usar raw strings `r"..."` o, mejor, `Path`.

In [13]:
print(len("\n"), len("\\"))
print(r"C:\nuevo\datos")
print("Neuqu\u00e9n")

1 1
C:\nuevo\datos
Neuquén


## JSON y caracteres no ASCII

| | resultado |
|---|---|
| `json.dumps(d)` | escapa: `"Neuqu\u00e9n"` (ASCII puro) |
| `json.dumps(d, ensure_ascii=False)` | legible: `"Neuquén"` |
| `indent=2` | una clave por línea |

Las dos formas representan los mismos datos; al leer, `json.load` convierte los `\uXXXX` automáticamente.

In [55]:
import json

medicion = {"estacion": "Neuquén", "temperatura": 19.2, "unidad": "°C"}
print(json.dumps(medicion))
print(json.dumps(medicion, ensure_ascii=False))

{"estacion": "Neuqu\u00e9n", "temperatura": 19.2, "unidad": "\u00b0C"}
{"estacion": "Neuquén", "temperatura": 19.2, "unidad": "°C"}


## La receta para archivos JSON

JSON (RFC 8259), el estándar oficial de Internet (publicado en diciembre de 2017) que define la sintaxis y las reglas del formato JSON es UTF-8, así que usamos siempre 

`ensure_ascii=False`

In [56]:
from pathlib import Path

salida = Path("salida")
salida.mkdir(exist_ok=True)
ruta = salida / "medicion.json"

with open(ruta, "w", encoding="utf-8") as archivo:
    json.dump(medicion, archivo, ensure_ascii=False, indent=2)

with open(ruta, "r", encoding="utf-8") as archivo:
    print(json.load(archivo))

{'estacion': 'Neuquén', 'temperatura': 19.2, 'unidad': '°C'}


In [24]:
from pathlib import Path

salida = Path("salida")
salida.mkdir(exist_ok=True)
ruta = salida / "medicion.json"

with open(ruta, "w", encoding="utf-8") as archivo:
    json.dump(medicion, archivo, ensure_ascii=True, indent=2)

with open(ruta, "r", encoding="utf-8") as archivo:
    print(json.load(archivo))

{'estacion': 'Neuquén', 'temperatura': 19.2, 'unidad': '°C'}


Si no ponemos `encoding=`, `open()` usa el del sistema (en Windows `cp1252`):

---
# Parte 2 — Fechas y horas como texto

En los archivos del SMN la fecha viene como `27-sep-2026` y la hora como `12:00`.
Para hacer cuentas (ordenar, restar, comparar) necesitamos convertir ese texto a un objeto `datetime`, y al guardar, volver a texto.

- `strptime` (string parse time): texto  a `datetime`
- `strftime` (string format time): `datetime` a texto

Los dos usan códigos de formato con `%`. Vienen de la función `strftime` de C, estandarizada por POSIX, por eso son los mismos en diversas plataformas.

In [57]:
from datetime import datetime

ahora = datetime.now() #datetime.datetime

In [58]:
print(ahora)
type(ahora)

2026-10-06 13:18:11.043774


datetime.datetime

## Códigos de formato más usados

| código | significado | ejemplo |
|---|---|---|
| `%Y` | año, 4 dígitos | `2026` |
| `%y` | año, 2 dígitos | `26` |
| `%m` | mes, 2 dígitos | `09` |
| `%d` | día del mes, 2 dígitos | `27` |
| `%H` | hora 00–23 | `14` |
| `%M` | minutos | `05` |
| `%S` | segundos | `09` |
| `%j` | día del año 001–366 | `270` |
| `%a` / `%A` | día de la semana abreviado / completo | `Sun` / `Sunday` |
| `%b` / `%B` | mes abreviado / completo | `Sep` / `September` |
| `%z` | diferencia con UTC | `-0300` |
| `%Z` | nombre de zona horaria | `UTC` |

In [60]:
#from datetime import datetime, date, timedelta
d = datetime(2026, 9, 28, 14, 5)  # (datetime.datetime)

print(d.strftime("%d/%m/%Y"))
print(d.strftime("%Y-%m-%d %H:%M:%S"))
print(d.strftime("día %j del año"))
print(d.strftime("%A %d de %B"))

28/09/2026
2026-09-28 14:05:00
día 271 del año
Monday 28 de September


In [61]:
d.isoformat()

'2026-09-28T14:05:00'

## Formatos estándar

### ISO 8601 — el que hay que usar

`AAAA-MM-DDTHH:MM:SS` (la `T` separa fecha y hora; se puede agregar la zona: `-03:00` o `Z` para UTC).
Ventajas:
- no depende del idioma,
- ordenar alfabéticamente es ordenar cronológicamente.

`datetime.datetime` lo tiene incorporado: `.isoformat()` y `.fromisoformat()`.

In [62]:
#from datetime import datetime, date, timedelta

d = datetime(2026, 9, 30, 14, 5, 9)  # (datetime.datetime)

print(d.isoformat())

2026-09-30T14:05:09


In [33]:
print(d.date().isoformat()) # fecha solamente

2026-09-30T14:05:09
2026-09-30


In [ ]:
#print(datetime.fromisoformat("2026-09-30T14:05:09"))

In [63]:
# Ordenar los strings ISO da el orden cronológico
fechas_iso = ["2026-10-01", "2025-12-31", "2026-09-27"]
print(sorted(fechas_iso))

# Con día/mes/año, no
fechas_dmy = ["01/10/2026", "31/12/2025", "27/09/2026"]
print(sorted(fechas_dmy))

['2025-12-31', '2026-09-27', '2026-10-01']
['01/10/2026', '27/09/2026', '31/12/2025']


### Otros formatos que van a encontrar

| nombre | formato `strftime` | ejemplo |
|---|---|---|
| ISO 8601 fecha | `%Y-%m-%d` | `2026-09-27` |
| ISO 8601 fecha y hora | `%Y-%m-%dT%H:%M:%S` | `2026-09-27T14:05:09` |
| ISO 8601 compacto (nombres de archivo) | `%Y%m%dT%H%M%S` | `20260927T140509` |
| RFC 2822 (mails, HTTP) | `%a, %d %b %Y %H:%M:%S %z` | `Sun, 27 Sep 2026 14:05:09 -0300` |
| `asctime` de C | `%c` en locale C | `Sun Sep 27 14:05:09 2026` |

Además existe el tiempo POSIX (o *Unix timestamp*): la cantidad de segundos desde el `1970-01-01 00:00:00 UTC`.
No es texto legible, pero es un solo número, cómodo para guardar y restar.

In [31]:
t = d.timestamp()
print(t)
print(datetime.fromtimestamp(t))
print(datetime.fromtimestamp(0))   # el "epoch" (en hora local)

1790615109.0
2026-09-28 14:05:09
1969-12-31 21:00:00


## De texto a fecha: `strptime`

El formato tiene que coincidir **exactamente** con el texto.

In [ ]:
print(datetime.strptime("27/09/2026 14:05", "%d/%m/%Y %H:%M"))
print(datetime.strptime("20260927", "%Y%m%d"))

In [ ]:
try:
    datetime.strptime("27/09/2026", "%Y-%m-%d")
except ValueError as e:
    print("ValueError:", e)

## Las fechas del SMN: `27-sep-2026`

El mes viene **abreviado en castellano**. Con `%b` en inglés, `sep` funciona de casualidad (`Sep`), pero `ago`, `dic`, `ene`, `abr` no:

In [ ]:
print(datetime.strptime("27-sep-2026", "%d-%b-%Y"))

for texto in ["15-ago-2026", "25-dic-2026"]:
    try:
        datetime.strptime(texto, "%d-%b-%Y")
    except ValueError as e:
        print("ValueError:", e)

Solución sin depender del locale del sistema: traducir el mes nosotros con un diccionario.

In [ ]:
MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
         "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12}

def leer_fecha_smn(fecha, hora):
    dia, mes, anio = fecha.split("-")
    h, m = hora.split(":")
    return datetime(int(anio), MESES[mes.lower()], int(dia), int(h), int(m))

f = leer_fecha_smn("15-ago-2026", "12:00")
print(f)
print(f.isoformat())

## Cuentas con fechas: `timedelta`

In [ ]:
inicio = datetime.fromisoformat("2026-09-27T12:00")
fin = datetime.fromisoformat("2026-09-29T18:30")

diferencia = fin - inicio
print(diferencia, type(diferencia))
print(diferencia.total_seconds() / 3600, "horas")

print(inicio + timedelta(days=7))

## Fechas y JSON

JSON no tiene tipo fecha: `json.dumps` no sabe qué hacer con un `datetime`.
Lo estándar es guardarla como **string ISO 8601**.

In [36]:
obs = {"estacion": "Ñorquinco", "fecha": datetime(2026, 9, 28, 12, 0)}

try:
    json.dumps(obs)
except TypeError as e:
    print(type(e).__name__,'---', e)

TypeError --- Object of type datetime is not JSON serializable


In [ ]:
obs["fecha"] = obs["fecha"].isoformat()
texto = json.dumps(obs, ensure_ascii=False)
print(texto)

leido = json.loads(texto)
print(datetime.fromisoformat(leido["fecha"]))